# 02 — Exploratory Data Analysis
**Project:** Charlotte NPA Housing Affordability Analysis  
**Target Variable (Regression):** Median Home Sales Price (2023, dollars)  
**Target Variable (Classification):** Displacement Risk (engineered in this notebook)  

## Purpose
This notebook loads the merged `npa_features` table from the SQLite database and performs a full exploratory analysis to support the regression and classification modeling notebooks that follow. The goals are:

1. Confirm data integrity (shape, types, nulls, year alignment)
2. Profile the target variable `home_sales_price` (distribution, outliers, transformation needs)
3. Profile each of the 8 candidate predictors
4. Quantify feature-target relationships through correlation, scatter analysis, and rank tables
5. Identify multicollinearity issues among predictors
6. Engineer the `displacement_risk` classification label using the 30 percent affordability rule
7. Recommend which features to drop, transform, or engineer further before modeling
8. Save a clean, model-ready table back to the database

## Important Project Note
Our target was originally Median Home Value but is now Median Home Sales Price. Sales price is a transactional measure that reflects what buyers actually paid in 2023, which is more responsive to short-term market pressure than appraised value. This makes it a stronger signal for displacement risk modeling but it also introduces more variance because not every NPA had the same transaction volume.

In [ ]:
import sqlite3
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

# Dynamic base path so any partner can run this without editing
base = os.path.abspath(os.path.join(os.getcwd(), '..'))
db_path = os.path.join(base, 'data', 'charlotte_housing.db')

conn = sqlite3.connect(db_path)
print(f'Connected to: {db_path}')

## 1. Load and Inspect the Merged Table

In [ ]:
df = pd.read_sql('SELECT * FROM npa_features', conn)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Column types
print('Data types:')
print(df.dtypes)

In [ ]:
# Confirm the year alignment we documented in notebook 01
year_cols = [c for c in df.columns if c.endswith('_year')]
print('Data year by feature:')
print(df[year_cols].drop_duplicates().to_string(index=False))
print('\nAll features are 2023 except voter_participation which is 2025.')

## 2. Missing Value Audit
We need to know which columns have nulls, how many, and whether the rows missing the target are the same rows missing predictors. NPAs missing the target have to be dropped because we cannot train or evaluate on them.

In [ ]:
feature_cols = [
    'home_sales_price',
    'tree_canopy_pct',
    'bachelors_pct',
    'water_consumption_gpd',
    'median_rent',
    'absenteeism_pct',
    'household_income',
    'housing_size_sqft',
    'voter_participation_pct',
]

miss = df[feature_cols].isna().sum().to_frame('null_count')
miss['null_pct'] = (miss['null_count'] / len(df) * 100).round(2)
miss = miss.sort_values('null_count', ascending=False)
miss

In [ ]:
# How many NPAs have a missing target?
missing_target = df['home_sales_price'].isna().sum()
print(f'NPAs with missing home_sales_price: {missing_target} of {len(df)}')
print(f'Pct of NPAs with usable target: {(len(df) - missing_target) / len(df) * 100:.2f}%')

# Drop rows with no target. Modeling requires y.
df_model = df.dropna(subset=['home_sales_price']).copy()
print(f'\nWorking dataset shape after dropping null targets: {df_model.shape}')

In [ ]:
# Re-audit nulls on the modeling subset
miss2 = df_model[feature_cols].isna().sum().to_frame('null_count')
miss2['null_pct'] = (miss2['null_count'] / len(df_model) * 100).round(2)
miss2.sort_values('null_count', ascending=False)

## 3. Target Variable Profile: `home_sales_price`
Before any modeling we need to understand the shape, range, and tail behavior of the target. Housing data is almost always right-skewed, which usually argues for a log transform when fitting OLS.

In [ ]:
tgt = df_model['home_sales_price']
summary = pd.DataFrame({
    'metric': ['count', 'mean', 'std', 'min', '25%', 'median', '75%', '95%', '99%', 'max', 'skew', 'kurtosis'],
    'value': [
        tgt.count(),
        tgt.mean(),
        tgt.std(),
        tgt.min(),
        tgt.quantile(0.25),
        tgt.median(),
        tgt.quantile(0.75),
        tgt.quantile(0.95),
        tgt.quantile(0.99),
        tgt.max(),
        tgt.skew(),
        tgt.kurtosis(),
    ]
})
summary

In [ ]:
# Distribution: raw vs log
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(tgt.dropna(), bins=40, edgecolor='black')
axes[0].set_title('home_sales_price — raw')
axes[0].set_xlabel('Median Home Sales Price ($)')
axes[0].set_ylabel('NPA count')
axes[0].axvline(tgt.median(), color='red', linestyle='--', label=f'Median ${tgt.median():,.0f}')
axes[0].axvline(tgt.mean(), color='orange', linestyle='--', label=f'Mean ${tgt.mean():,.0f}')
axes[0].legend()

log_tgt = np.log(tgt.dropna())
axes[1].hist(log_tgt, bins=40, edgecolor='black', color='seagreen')
axes[1].set_title('home_sales_price — log transformed')
axes[1].set_xlabel('log(Median Home Sales Price)')
axes[1].set_ylabel('NPA count')

plt.tight_layout()
plt.show()

print(f'Raw skew: {tgt.skew():.3f}')
print(f'Log skew: {log_tgt.skew():.3f}')

In [ ]:
# Boxplot to identify high-end outliers
fig, ax = plt.subplots(figsize=(12, 3))
ax.boxplot(tgt.dropna(), vert=False)
ax.set_xlabel('Median Home Sales Price ($)')
ax.set_title('Outlier scan: home_sales_price')
plt.tight_layout()
plt.show()

# Top 10 highest priced NPAs
top10 = df_model.nlargest(10, 'home_sales_price')[['npa_id', 'home_sales_price', 'household_income']]
print('\nTop 10 NPAs by home sales price:')
print(top10.to_string(index=False))

**Target takeaway.** The raw distribution is heavily right-skewed because a small number of NPAs have very high median sales prices. A log transform brings the distribution closer to normal, which matters for OLS assumptions. We will model both the raw target and `log(home_sales_price)` in notebook 03 and pick whichever gives a better train and test fit with healthier residual diagnostics.

## 4. Predictor Profiles
Numeric summary plus a histogram for each of the 8 candidate features. Two passes here: a stats table for fast scanning and a small-multiples grid for shape inspection.

In [ ]:
predictors = [c for c in feature_cols if c != 'home_sales_price']
df_model[predictors].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flat, predictors):
    ax.hist(df_model[col].dropna(), bins=30, edgecolor='black')
    ax.set_title(col)
    ax.set_xlabel('')
plt.suptitle('Predictor distributions', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 5. Feature-Target Relationships

In [ ]:
# Pearson and Spearman correlations of every feature vs the target
rows = []
for col in predictors:
    sub = df_model[[col, 'home_sales_price']].dropna()
    pearson = sub[col].corr(sub['home_sales_price'], method='pearson')
    spearman = sub[col].corr(sub['home_sales_price'], method='spearman')
    rows.append({'feature': col, 'pearson_r': pearson, 'spearman_rho': spearman, 'n': len(sub)})

corr_table = pd.DataFrame(rows).sort_values('pearson_r', key=abs, ascending=False).reset_index(drop=True)
corr_table

In [ ]:
# Full correlation heatmap including the target
corr = df_model[feature_cols].corr(method='pearson')

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            cbar_kws={'shrink': 0.8})
plt.title('Pearson correlation matrix — target + 8 predictors')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter every predictor against the target so we can eyeball linearity
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, col in zip(axes.flat, predictors):
    sub = df_model[[col, 'home_sales_price']].dropna()
    ax.scatter(sub[col], sub['home_sales_price'], alpha=0.4, s=20)
    r = sub[col].corr(sub['home_sales_price'])
    ax.set_title(f'{col}  (r = {r:.2f})')
    ax.set_xlabel(col)
    ax.set_ylabel('home_sales_price')
plt.suptitle('Each predictor vs home_sales_price', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 6. Multicollinearity Check (VIF)
Several of these features are likely to move together. Household income, bachelor's percentage, and median rent in particular tend to be tightly correlated in Charlotte NPAs. VIF tells us whether any single predictor can be linearly explained by the others, which would inflate OLS coefficient variance and make interpretation unreliable.

Rule of thumb: VIF below 5 is fine, 5 to 10 is a warning, above 10 is a real multicollinearity problem.

In [ ]:
vif_df = df_model[predictors].dropna().copy()
X_vif = add_constant(vif_df)
vif = pd.DataFrame({
    'feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
vif = vif[vif['feature'] != 'const'].sort_values('VIF', ascending=False).reset_index(drop=True)
vif

## 7. Engineer the Classification Target: `displacement_risk`
Standard housing affordability rule: a household should not spend more than 30 percent of gross income on housing. We compute the implied monthly mortgage payment for the median home in each NPA assuming a 30 year fixed loan at 7 percent interest, annualize it, and compare it to 30 percent of the median household income in that NPA.

An NPA is flagged At Risk (1) when implied annual housing cost exceeds 30 percent of median household income. Otherwise it is Stable (0). This produces a binary label tied directly to a real housing policy benchmark used by HUD.

In [ ]:
# Mortgage payment factor for a 30-year loan at 7% APR
# Monthly payment per dollar borrowed = r(1+r)^n / ((1+r)^n - 1)
r = 0.07 / 12
n = 360
monthly_factor = (r * (1 + r) ** n) / ((1 + r) ** n - 1)
print(f'Monthly payment factor (30yr @ 7%): {monthly_factor:.6f}')

df_model['implied_monthly_cost'] = df_model['home_sales_price'] * monthly_factor
df_model['implied_annual_cost'] = df_model['implied_monthly_cost'] * 12
df_model['affordable_annual_housing'] = df_model['household_income'] * 0.30
df_model['cost_to_affordable_ratio'] = df_model['implied_annual_cost'] / df_model['affordable_annual_housing']

df_model['displacement_risk'] = (df_model['implied_annual_cost'] > df_model['affordable_annual_housing']).astype(int)

In [ ]:
# Class balance check
vc = df_model['displacement_risk'].value_counts().sort_index()
vc.index = ['Stable (0)', 'At Risk (1)']
print('Class counts:')
print(vc)
print(f'\nAt Risk share: {df_model["displacement_risk"].mean() * 100:.2f}%')

In [ ]:
# Visualize the affordability ratio. Anything to the right of 1.0 is At Risk.
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_model['cost_to_affordable_ratio'].dropna(), bins=40, edgecolor='black')
ax.axvline(1.0, color='red', linestyle='--', linewidth=2, label='At Risk threshold (ratio = 1.0)')
ax.set_xlabel('Implied housing cost / 30% of household income')
ax.set_ylabel('NPA count')
ax.set_title('Distribution of affordability ratio across NPAs')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Sales price distribution split by class to show the label is meaningful
fig, ax = plt.subplots(figsize=(10, 5))
for label, color in [(0, 'steelblue'), (1, 'firebrick')]:
    sub = df_model[df_model['displacement_risk'] == label]['home_sales_price']
    ax.hist(sub.dropna(), bins=30, alpha=0.6, color=color,
            label=f'{"At Risk" if label == 1 else "Stable"} (n={len(sub)})', edgecolor='black')
ax.set_xlabel('Median Home Sales Price ($)')
ax.set_ylabel('NPA count')
ax.set_title('home_sales_price by displacement_risk class')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Feature Engineering Recommendations

This section is the bridge between EDA and modeling. Based on the diagnostics above, here is what we propose to do with each feature before it goes into notebook 03 and notebook 04.

### Drop
- **`voter_participation_pct`** is from 2025 while every other feature is 2023. The two year gap means a 2025 turnout pattern is being asked to explain a 2023 sales price. We document this in the limitations section regardless, but if we drop it we eliminate a confound. We will compare model performance with and without it in notebook 03 and decide based on the test R squared and overfit gap.

### Transform
- **`home_sales_price`**: apply a log transform for the regression model. The raw target is right skewed and the log version is much closer to normal, which improves OLS residual behavior.
- **`water_consumption_gpd` and `housing_size_sqft`**: check for log or square root transforms in notebook 03 if the scatter shows curvature. Both have wide ranges and weak linear correlation with the target.

### Engineer
- **`rent_to_income_ratio` = `median_rent` * 12 / `household_income`**: a continuous version of housing burden that captures the same idea as displacement risk but as a number rather than a label. Useful as an additional regression predictor.
- **`income_per_sqft` = `household_income` / `housing_size_sqft`**: proxies how much earning power exists per unit of housing in an NPA. Could expose nonlinear relationships the raw features miss.

### Watch for multicollinearity
- The VIF table shows whether `bachelors_pct`, `household_income`, and `median_rent` are too tangled up. If any feature has a VIF above 10 we drop the one with the weakest correlation with the target and refit.

### Keep as is
- `tree_canopy_pct`, `bachelors_pct`, `absenteeism_pct`, `household_income`, `median_rent` enter the model in their raw form unless multicollinearity forces a change.

## 9. Save the Model-Ready Table
We write the cleaned dataset (target dropped where missing, displacement risk engineered, ratios computed) back to the database as `npa_features_model`. Notebooks 03 and 04 read from this table, not from the raw merged table.

In [ ]:
# Engineer the two derived features so they are saved alongside everything else
df_model['rent_to_income_ratio'] = (df_model['median_rent'] * 12) / df_model['household_income']
df_model['income_per_sqft'] = df_model['household_income'] / df_model['housing_size_sqft']
df_model['log_home_sales_price'] = np.log(df_model['home_sales_price'])

save_cols = (
    ['npa_id', 'home_sales_price', 'log_home_sales_price', 'displacement_risk',
     'cost_to_affordable_ratio', 'rent_to_income_ratio', 'income_per_sqft']
    + predictors
)
df_model[save_cols].to_sql('npa_features_model', conn, if_exists='replace', index=False)
print(f'npa_features_model saved: {len(df_model)} rows, {len(save_cols)} columns')
print(f'\nColumns saved: {save_cols}')

In [ ]:
# Sanity check: read it back and confirm
check = pd.read_sql('SELECT * FROM npa_features_model LIMIT 5', conn)
check

In [ ]:
conn.close()
print('Connection closed. EDA complete.')
print('Notebooks 03 (regression) and 04 (classification) should both read from npa_features_model.')